# 03 · Barcodes y Persistence Landscapes por Grupo
Notebook autocontenido y minimalista para dos salidas:
- barcodes de persistencia para `aceptadores` y `no_aceptadores`
- persistence landscapes promedio por grupo
Definición de grupo: `aceptador` si `consume_rate >= 0.5`; en otro caso `no_aceptador`.
La serie completa por infante concatena todas las etapas afectivas disponibles y la PH se calcula sobre la nube 2D `valence-arousal`.
Si una nube grupal supera `500` puntos, el barcode grupal usa la aproximación `n_perm` de `ripser` para mantener el notebook liviano.


## 1. Setup

Aquí importamos las librerías y fijamos constantes del análisis.

- `STAGES` define el orden temporal de las mediciones afectivas dentro de cada día.
- `DATA_CANDIDATES` permite abrir cualquiera de los dos nombres posibles del Excel.
- `OUTPUT_DIR` es la carpeta donde se guardan las figuras finales.
- `NUM_LANDSCAPE_LAYERS` y `NUM_LANDSCAPE_STEPS` controlan la resolución de los landscapes.
- `MAX_POINTS_FOR_GROUP_PH` limita el tamaño efectivo de la nube grupal cuando `ripser` usa la aproximación por permutaciones.


In [ ]:
from pathlib import Path
import re
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from persim import PersLandscapeApprox
from ripser import ripser
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')  # Evita ruido visual de warnings no críticos.
BASE_DIR = Path.cwd()
DATA_CANDIDATES = [
    BASE_DIR / 'DFBabyFaceValidacion.xlsx',
    BASE_DIR / 'DataFrameBabyFaceValidacion.xlsx',
]
OUTPUT_DIR = BASE_DIR / '03_persistent_homology_outputs_minimal'
OUTPUT_DIR.mkdir(exist_ok=True)

# Orden fijo de etapas para reconstruir la serie completa por infante.
STAGES = [
    ('inicio', 'valence_inicio', 'arousal_inicio'),
    ('p1_before', 'valence_p1_before', 'arousal_p1_before'),
    ('p1_during', 'valence_p1_during', 'arousal_p1_during'),
    ('p1_after', 'valence_p1_after', 'arousal_p1_after'),
    ('p2_before', 'valence_p2_before', 'arousal_p2_before'),
    ('p2_during', 'valence_p2_during', 'arousal_p2_during'),
    ('p2_after', 'valence_p2_after', 'arousal_p2_after'),
    ('final', 'valence_final', 'arousal_final'),
]
STAGE_ORDER = {stage: idx for idx, (stage, _, _) in enumerate(STAGES)}
MISSING_TOKENS = {'', '-', 'NA', 'NA ', 'N/A', 'nan', 'NaN', 'FIT_FAILED', 'FIT FAILED', 'FIND_FAILED', 'S/I', None}
EPSILON = 1e-10
NUM_LANDSCAPE_LAYERS = 5
NUM_LANDSCAPE_STEPS = 500
MAX_POINTS_FOR_GROUP_PH = 500
GROUP_COLORS = {'aceptador': '#1b9e77', 'no_aceptador': '#d95f02'}
DIM_COLORS = {0: '#1f78b4', 1: '#e31a1c'}


## 2. Funciones auxiliares

Estas funciones hacen el trabajo técnico del notebook:

- limpian el Excel y convierten columnas a numérico,
- calculan la tasa de consumo por infante para asignar grupo,
- reorganizan la información a formato largo para reconstruir la serie afectiva completa,
- calculan diagramas de persistencia H0 y H1,
- eliminan features infinitas o prácticamente nulas,
- convierten diagramas en persistence landscapes promedio.


In [ ]:
def id_sort_key(value):
    # Ordena IDs tipo ID1, ID2, ..., ID20 por su número interno.
    match = re.search(r'(\d+)', str(value))
    return (int(match.group(1)), str(value)) if match else (10**9, str(value))
def parse_numeric(x):
    # Convierte strings numéricos y tokens mixtos del Excel a float/NaN.
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    if s in MISSING_TOKENS:
        return np.nan
    try:
        return float(s.replace(',', '.'))
    except Exception:
        return np.nan
def parse_day_number(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in MISSING_TOKENS:
        return np.nan
    match = re.search(r'\d+(?:\.\d+)?', s)
    return float(match.group(0)) if match else np.nan
def resolve_data_path():
    # Usa el primer archivo de datos disponible entre los nombres esperados.
    for path in DATA_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError('No se encontró el Excel de entrada.')
def load_and_clean(path):
    # Lee el Excel y normaliza las columnas que usaremos en el análisis topológico.
    df = pd.read_excel(path).copy()
    df['id'] = df['id'].astype(str).str.strip()
    df['dia_num'] = df['dia'].apply(parse_day_number)
    binary_cols = ['rechaza', 'llora', 'toca', 'prueba', 'consume']
    va_cols = [c for c in df.columns if c.startswith('valence') or c.startswith('arousal')]
    for col in binary_cols + va_cols:
        if col in df.columns:
            df[col] = df[col].apply(parse_numeric)
    return df
def classify_infants(df):
    # Los faltantes binarios se interpretan como ausencia del evento para el promedio por infante.
    rates_df = df.copy()
    for col in ['consume', 'rechaza', 'prueba', 'llora']:
        rates_df[col] = rates_df[col].fillna(0.0)
    grouped = (
        rates_df.groupby('id')
        .agg(consume_rate=('consume', 'mean'), n_registros=('dia_num', 'count'))
        .reset_index()
    )
    # Regla de grupo: aceptador si consume en al menos la mitad de sus registros.
    grouped['grupo'] = np.where(grouped['consume_rate'] >= 0.5, 'aceptador', 'no_aceptador')
    return grouped.sort_values('id', key=lambda s: s.map(id_sort_key)).reset_index(drop=True)
def build_long_series(df):
    # Pasa de formato ancho a formato largo para concatenar todas las etapas afectivas.
    rows = []
    for stage, vcol, acol in STAGES:
        tmp = df[['id', 'dia_num', vcol, acol]].copy()
        tmp.columns = ['id', 'dia_num', 'valence', 'arousal']
        tmp['stage'] = stage
        tmp['stage_order'] = STAGE_ORDER[stage]
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)
    long_df = long_df.sort_values(['id', 'dia_num', 'stage_order'], kind='stable').reset_index(drop=True)
    # Solo conservamos puntos donde valence y arousal existen al mismo tiempo.
    long_df = long_df.dropna(subset=['valence', 'arousal']).copy()
    # Escalamos globalmente para que todos los infantes y grupos queden en la misma escala.
    scaler = StandardScaler()
    long_df[['valence_scaled', 'arousal_scaled']] = scaler.fit_transform(long_df[['valence', 'arousal']])
    return long_df
def compute_diagrams(points):
    # Calcula diagramas Vietoris-Rips en H0 y H1 sobre la nube 2D del grupo o del infante.
    if len(points) == 0:
        return [np.empty((0, 2)), np.empty((0, 2))]
    if len(points) == 1:
        return [np.array([[0.0, np.inf]], dtype=float), np.empty((0, 2))]
    kwargs = {'maxdim': 1}
    # Si la nube grupal es grande, usamos una aproximación para mantener el notebook liviano.
    if len(points) > MAX_POINTS_FOR_GROUP_PH:
        kwargs['n_perm'] = MAX_POINTS_FOR_GROUP_PH
    dgms = ripser(points, **kwargs)['dgms']
    if len(dgms) == 1:
        dgms.append(np.empty((0, 2)))
    return [np.asarray(dgm, dtype=float) for dgm in dgms[:2]]
def finite_positive_diagram(diagram):
    # Quitamos features infinitas o de persistencia prácticamente cero para simplificar la lectura.
    if diagram.size == 0:
        return np.empty((0, 2))
    finite_mask = np.isfinite(diagram[:, 0]) & np.isfinite(diagram[:, 1])
    filtered = diagram[finite_mask]
    if filtered.size == 0:
        return np.empty((0, 2))
    persistence = filtered[:, 1] - filtered[:, 0]
    return filtered[persistence > EPSILON]
def build_landscape_input(diagram, homology_dim):
    # PersLandscapeApprox espera una lista indexada por dimensión de homología.
    if homology_dim == 0:
        return [diagram]
    return [np.empty((0, 2)), diagram]
def landscape_range(diagrams):
    # Fija una grilla común de birth-death para promediar landscapes entre infantes.
    nonempty = [diagram for diagram in diagrams if diagram.size > 0]
    if not nonempty:
        return 0.0, 1.0
    stacked = np.vstack(nonempty)
    start = float(np.min(stacked[:, 0]))
    stop = float(np.max(stacked[:, 1]))
    if not np.isfinite(stop) or stop <= start + EPSILON:
        stop = start + 1.0
    return start, stop
def landscape_values(diagram, homology_dim, start, stop, num_steps=NUM_LANDSCAPE_STEPS, num_layers=NUM_LANDSCAPE_LAYERS):
    # Convierte un diagrama individual en una matriz de landscapes k x t.
    if diagram.size == 0:
        return np.zeros((num_layers, num_steps), dtype=float)
    landscape = PersLandscapeApprox(
        dgms=build_landscape_input(diagram, homology_dim),
        hom_deg=homology_dim,
        start=start,
        stop=stop,
        num_steps=num_steps,
    )
    values = np.asarray(landscape.values, dtype=float)
    if values.ndim == 1:
        values = values.reshape(1, -1)
    output = np.zeros((num_layers, num_steps), dtype=float)
    if values.size > 0:
        n_layers = min(num_layers, values.shape[0])
        output[:n_layers, :] = values[:n_layers, :]
    return output
def plot_barcode(ax, diagram, title, color):
    # Dibuja cada intervalo birth-death como una barra horizontal.
    if diagram.size == 0:
        ax.text(0.5, 0.5, 'Sin barras finitas', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        ax.set_yticks([])
        return
    persistence = diagram[:, 1] - diagram[:, 0]
    order = np.argsort(persistence)[::-1]
    sorted_diagram = diagram[order]
    for idx, (birth, death) in enumerate(sorted_diagram):
        ax.hlines(y=idx, xmin=birth, xmax=death, color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Escala')
    ax.set_ylabel('Barra')
    ax.grid(alpha=0.2)


## 3. Carga, limpieza y armado de grupos

En esta celda ocurren tres cosas:

1. se carga el Excel y se limpian columnas numéricas,
2. se calcula `consume_rate` por infante para decidir si pertenece a `aceptador` o `no_aceptador`,
3. se construyen dos niveles de diagramas:
   - diagramas grupales, usando todos los puntos de todos los infantes del grupo,
   - diagramas por infante, que luego sirven para promediar los landscapes dentro de cada grupo.


In [ ]:
data_path = resolve_data_path()
df = load_and_clean(data_path)
infant_groups = classify_infants(df)

# Reconstruimos la serie completa de valence-arousal de todos los infantes.
long_series = build_long_series(df)
# Añadimos la etiqueta de grupo a cada punto afectivo.
long_series = long_series.merge(infant_groups[['id', 'grupo']], on='id', how='left')
ordered_ids = infant_groups['id'].tolist()
groups_by_id = dict(zip(infant_groups['id'], infant_groups['grupo']))

# Nube grupal: mezcla de todos los puntos de todos los infantes del mismo grupo.
group_point_clouds = {}
for group in ['aceptador', 'no_aceptador']:
    sub = long_series[long_series['grupo'] == group]
    group_point_clouds[group] = sub[['valence_scaled', 'arousal_scaled']].to_numpy(dtype=float)

# Diagramas de persistencia por grupo para los barcodes finales.
group_diagrams = {}
for group, points in group_point_clouds.items():
    group_diagrams[group] = [finite_positive_diagram(dgm) for dgm in compute_diagrams(points)]

# Diagramas por infante para poder promediar landscapes dentro de cada grupo.
infant_diagrams = {}
for id_value, sub in long_series.groupby('id'):
    points = sub[['valence_scaled', 'arousal_scaled']].to_numpy(dtype=float)
    infant_diagrams[id_value] = [finite_positive_diagram(dgm) for dgm in compute_diagrams(points)]
print(f'Archivo cargado: {data_path.name}')
print('Conteos por grupo:')
print(infant_groups.groupby('grupo').agg(n_infantes=('id', 'count'), consume_rate_media=('consume_rate', 'mean')).reset_index().to_string(index=False))
print('\nInfantes y tasas de consumo:')
print(infant_groups[['id', 'grupo', 'consume_rate']].to_string(index=False))


## 4. Barcodes por grupo

Aquí cada barra representa una feature topológica que nace en una escala y muere en otra.

- `H0` resume componentes conectadas.
- `H1` resume ciclos o huecos.

La figura compara directamente `aceptadores` vs. `no_aceptadores` usando la nube agregada de cada grupo.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Filas = grupos; columnas = dimensiones de homología.
for row_idx, group in enumerate(['aceptador', 'no_aceptador']):
    for dim in (0, 1):
        plot_barcode(
            axes[row_idx, dim],
            group_diagrams[group][dim],
            title=f"{group} · H{dim}",
            color=DIM_COLORS[dim],
        )
fig.suptitle('Barcodes de persistencia por grupo', fontsize=16)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'barcodes_por_grupo.png', dpi=180, bbox_inches='tight')
plt.show()


## 5. Persistence landscapes promedio por grupo

El barcode es útil para inspección visual, pero no es tan fácil de promediar entre infantes. Por eso aquí convertimos cada diagrama individual en un `landscape` y luego promediamos esos landscapes dentro de cada grupo.

En la figura:

- la primera fila corresponde a `H0`,
- la segunda fila corresponde a `H1`,
- las tres columnas muestran las primeras tres capas del landscape promedio.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=False, sharey=False)

for dim in (0, 1):
    # La grilla se fija usando todos los diagramas individuales de esa dimensión.
    diagrams_for_dim = [infant_diagrams[id_value][dim] for id_value in ordered_ids]
    start, stop = landscape_range(diagrams_for_dim)
    x_grid = np.linspace(start, stop, NUM_LANDSCAPE_STEPS)

    mean_by_group = {}
    for group in ['aceptador', 'no_aceptador']:
        # Reunimos landscapes individuales del grupo y luego los promediamos punto a punto.
        arrays = []
        for id_value in ordered_ids:
            if groups_by_id[id_value] != group:
                continue
            arrays.append(landscape_values(infant_diagrams[id_value][dim], dim, start, stop))
        mean_by_group[group] = np.mean(np.stack(arrays, axis=0), axis=0)

    for layer_idx in range(3):
        ax = axes[dim, layer_idx]
        for group in ['aceptador', 'no_aceptador']:
            ax.plot(
                x_grid,
                mean_by_group[group][layer_idx],
                color=GROUP_COLORS[group],
                linewidth=2,
                label=group,
            )
        ax.set_title(f"H{dim} · Landscape {layer_idx + 1}")
        ax.set_xlabel('t')
        ax.set_ylabel('lambda_k(t)')
        ax.grid(alpha=0.25)
        if dim == 0 and layer_idx == 0:
            ax.legend()

fig.suptitle('Persistence landscapes promedio por grupo', fontsize=16)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'landscapes_promedio_por_grupo.png', dpi=180, bbox_inches='tight')
plt.show()


Las figuras también quedan guardadas en `03_persistent_homology_outputs_minimal/`:

- `barcodes_por_grupo.png`
- `landscapes_promedio_por_grupo.png`
